# 36장 · 검색 노드 · 벡터 검색 함수와 LCEL 답변 체인을 그래프에 연결하기

검색 노드는 질문을 받아 검색 함수를 호출하고 찾은 문단을 상태에 씁니다. 다음 노드가 원래 질문과 문단을 함께 읽으면 검색과 생성을 분리해 시험할 수 있습니다.

## 노드가 반환할 부분 상태 만들기

`search_node(state)`는 상태에서 질문을 읽고, 찾은 글을 `documents` 키에 담아 반환합니다. 먼저 두 문장 중 질문에 적은 단어가 들어간 것만 골라 봅니다. 실제 벡터 검색기로 바꾸어도 다음 노드에 넘길 키 이름은 같습니다.

In [ ]:
def search_node(state):
    pages = ["사과는 과일이다.", "소금은 짜다."]
    return {"documents": [p for p in pages if state["question"] in p]}

print(search_node({"question": "사과"}))

{'documents': ['사과는 과일이다.']}


이 셀의 검색 기준은 문자열 포함 여부입니다. 결과의 documents 키와 목록 형태를 유지하면 안쪽의 검색만 3부에서 만든 벡터 검색 함수로 교체할 수 있습니다.

## 응용 실습 · 실제 검색기 연결하기

짝 활동에는 [실습 수첩](pre-mid/documents/graph/workshop.md)을 사용합니다. 과제 아래에서 풀이를 펼칠 수 있습니다.

## 검색 노드와 답변 노드의 입력 정하기

검색 함수는 질문 문자열 하나를 받아 문단 목록을 반환합니다. 답변 함수에는 원래 질문과 그 문단 목록이 함께 들어갑니다. 상태에 `question`을 남겨 두면 검색 뒤에도 답변 노드가 원래 질문을 읽을 수 있습니다.

A형 1.0판과 1.1판의 2쪽을 펼치세요. 30분과 45분은 모두 설명서에 있는 숫자입니다. 두 값 중 어느 것이 이 질문의 기준일에 맞는지는 모델 부르기 전에 결정할 수 있습니다. 직접 찾은 문서명·페이지·근거 한 문장을 적어 두세요.

3부에서 쓰던 세 PDF와 공용 검색기를 같은 폴더에서 엽니다. 교재 ZIP 안의 `pre-mid/documents/rag`, `graph`, `sources`가 함께 있어야 합니다. 실제 검색 셀은 3부 Chroma 색인·OpenAI 임베딩과 API 키를 사용합니다. 처음 실행한다면 노트북 셀에서 `%pip install "langgraph>=1,<2" "langchain-openai>=1,<2" "chromadb>=1,<2" pypdf openai python-dotenv streamlit`를 실행하고 커널을 다시 시작하세요. PDF 직접 읽기와 고정 입력 경로 실험은 키 없이 실행됩니다.

In [1]:
from pathlib import Path
import sys

# 통합 ZIP과 저장소에서 같은 자료 폴더를 찾습니다.
docs = next(p / "pre-mid" / "documents" for p in (Path.cwd(), *Path.cwd().parents)
            if (p / "pre-mid" / "documents" / "graph").is_dir())
sys.path.insert(0, str(docs / "graph"))
sys.path.insert(0, str(docs / "rag"))
from rag_lab import MANUALS, read_pages

# 세 설명서의 모든 페이지를 실제 PDF에서 읽습니다.
pages = [page for path in MANUALS for page in read_pages(path)]
by_id = {page["id"]: page for page in pages}
for page in pages:
    if page["model"] == "A" and page["page"] == 2:
        line = next(line for line in page["text"].split("\n") if "분마다" in line)
        print(page["id"], line)

recorder-a-1.0.pdf#p2 한 번에 길게 녹음하면 30분마다 새 파일이 열립니다. 파일이
recorder-a-1.1.pdf#p2 한 번에 길게 녹음하면 45분마다 새 파일이 열립니다. 파일이


1.0판 2쪽은 30분, 1.1판 2쪽은 45분입니다. 4월에는 1.1판이 적용됩니다. 50분 동안 45분 경계가 한 번 있으니 파일은 두 개이고 녹음은 이어집니다. 페이지 번호가 같아도 출처 파일을 버리면 답을 검증할 수 없습니다.

## connect에 검색 함수와 답변 함수 넘기기

원래 질문은 question, 검색 후보는 documents, 최종 답은 answer에 둡니다. 검색 노드는 질문을 보존한 채 문단만 돌려줍니다. 생성 노드는 후보뿐 아니라 원래 질문도 읽어야 “몇 분?”과 “계속 녹음되나요?”를 함께 답할 수 있습니다.

`connect(search, generate)`는 실행할 두 함수를 받아 그래프를 만듭니다. 첫 노드는 질문 한 개를 `search`에 넘겨 문단 목록을 얻고 `{"documents": 목록}`만 반환합니다. 그래프가 이 값을 기존 상태에 합치므로 둘째 노드는 처음 질문과 방금 찾은 문단을 함께 읽습니다. 아래 고정 문단 실험에서는 전달을 확인하고, 실제 검색 실험에서 같은 자리에 3부의 함수를 끼웁니다.

In [2]:
from typing import TypedDict
from langgraph.graph import START, END, StateGraph

class RagState(TypedDict, total=False):
    question: str
    documents: list
    answer: str

def connect(search, generate):
    def search_node(state):
        return {"documents": search(state["question"])}
    def answer_node(state):
        return {"answer": generate(state["question"], state["documents"])}
    plan = StateGraph(RagState)
    plan.add_node("search", search_node)
    plan.add_node("answer", answer_node)
    plan.add_edge(START, "search")
    plan.add_edge("search", "answer")
    plan.add_edge("answer", END)
    return plan.compile()

이 연결에는 조건부 간선이 없습니다. 후보가 잘못 들어와도 답변 노드는 호출됩니다. 검색 문제와 답변 문제를 구분하는 기준선으로 먼저 남겨 둡니다.

## 고정한 문단 하나로 connect의 노드 연결 확인하기

검색기 대신 이미 손으로 고른 쪽을 넘겨 보세요. 생성 대신 도착한 자료 ID를 반환합니다. 이 단계에서는 검색 성능이나 답변 품질을 평가할 수 없고, 학생이 정한 상태 칸으로 자료가 전달되는지만 확인합니다.

In [3]:
q = "2027년 4월 15일에 A형을 50분간 녹음하면 파일 수와 녹음 상태는?"
target = by_id["recorder-a-1.1.pdf#p2"]
wiring = connect(lambda question: [target], lambda question, docs: docs[0]["id"])
result = wiring.invoke({"question": q})
print(result["question"])
print([p["id"] for p in result["documents"]])
print(result["answer"])

2027년 4월 15일에 A형을 50분간 녹음하면 파일 수와 녹음 상태는?
['recorder-a-1.1.pdf#p2']
recorder-a-1.1.pdf#p2


출력 마지막 줄이 파일명과 쪽 ID인 이유는 이 셀이 연결 확인에 쓰는 함수를 넘겼기 때문입니다. 이제 search와 generate만 실제 구현으로 바꿀 수 있습니다.

## 같은 질문·색인으로 LCEL과 그래프 실행하기

이전 색인 폴더에서 실행하면 같은 임베딩을 재사용합니다. 두 답변을 비교하기 전에 두 후보 목록부터 비교하세요. 답변의 표현은 모델 호출마다 달라질 수 있습니다.

In [4]:
import os
from dotenv import load_dotenv
from openai import OpenAI

# 8장에서 준비한 .env를 읽습니다. 키는 출력하지 않습니다.
load_dotenv()
use_live = bool(os.getenv("OPENAI_API_KEY"))
model_name = os.getenv("OPENAI_MODEL", "gpt-4.1-mini")
if use_live:
    client = OpenAI()

In [5]:
from rag_lab import open_index, retrieve, answer_question, answer_from_documents

if use_live:
    index = open_index()
    baseline = answer_question(index, q, k=4)
    graph = connect(lambda question: retrieve(index, question, k=4), answer_from_documents)
    traced = graph.invoke({"question": q})
    for label, item in [("LCEL", baseline), ("LangGraph", traced)]:
        print(label, [p["id"] for p in item["documents"]])
        print(item["answer"])
else:
    print("실제 색인과 모델 비교는 API 키를 준비한 뒤 실행하세요.")

실제 색인과 모델 비교는 API 키를 준비한 뒤 실행하세요.


두 후보 목록이 다르면 먼저 k, 색인, 입력 문장이 같은지 봅니다. 후보가 같은데 한 답이 30분이라고 했다면 원문과 선택한 판본이 일치하는지 답변 단계에서 다시 확인합니다.

## 질문은 그대로 두고 전달하는 문단만 바꾸기

50분 녹음 질문을 그대로 두고 입력 문단만 3월 판본으로 바꾸면 어떤 답이 나올까요? 먼저 학생 두 명이 각 판본의 직원이라고 가정합니다. 각자 받은 쪽의 파일 분할 간격만 보고 답한 뒤, 질문의 기준일을 공개합니다. 이제 같은 대조를 코드에서 재현합니다.

In [6]:
for candidate in [by_id["recorder-a-1.0.pdf#p2"], target]:
    print("전달할 판본", candidate["edition"])
    print(next(line for line in candidate["text"].split("\n") if "분마다" in line))

전달할 판본 1.0
한 번에 길게 녹음하면 30분마다 새 파일이 열립니다. 파일이
전달할 판본 1.1
한 번에 길게 녹음하면 45분마다 새 파일이 열립니다. 파일이


어느 답이든 받은 문단에서 숫자를 복사할 수 있습니다. 질문의 기준일과 문단의 적용 기간을 함께 확인해야 올바른 쪽을 고릅니다. 37장에서는 질문자가 고른 기종과 판본을 별도 필드로 받아서 생성 전에 확인합니다.

## 실습 · 쪽수만 남은 출처 복구하기

짝이 남긴 기록에는 “2쪽, 30분”만 있습니다. 어떤 기종·판본일 수 있는지 찾고, 4월의 A형 질문에 그대로 사용해도 되는지 판정하세요. 정확한 필드를 갖는 증거 카드를 만듭니다.

In [7]:
phrase_to_find = "30분마다"
# pages에서 page가 2이고 text에 이 문구가 든 쪽의 source와 edition을 찾으세요. target과 판본을 비교하세요.

<details><summary>모범 예시와 풀이</summary><pre><code>for item in pages:
    if item[&quot;page&quot;] == 2 and &quot;30분마다&quot; in item[&quot;text&quot;]:
        print(item[&quot;source&quot;], item[&quot;edition&quot;], item[&quot;page&quot;])
print(target[&quot;source&quot;], target[&quot;edition&quot;], target[&quot;page&quot;])</code></pre><p>30분은 A형 1.0판의 근거입니다. 4월에는 A형 1.1판 2쪽을 사용해야 하므로 구판 값을 그대로 인용하면 틀립니다.</p></details>

## 실습 · 검색 실패와 답변 실패 나누기

B형의 구체적인 질문 하나를 만들고 두 줄의 진단표를 작성하세요. 첫 실험은 잘못된 A형 페이지만 검색된 경우, 둘째는 B형 정답 쪽도 전달됐으나 답변자가 A형 값을 쓴 경우입니다. 어느 노드를 먼저 확인할지 근거와 함께 적습니다.

In [8]:
candidate_ids = ["recorder-a-1.1.pdf#p1", "recorder-b-1.0.pdf#p1"]
# by_id로 두 후보를 읽으세요. 첫 쪽만 줄 때와 둘 다 줄 때 B형 근거가 있는지 비교하세요.

<details><summary>모범 예시와 풀이</summary><pre><code>wrong = by_id[&quot;recorder-a-1.1.pdf#p1&quot;]
correct = by_id[&quot;recorder-b-1.0.pdf#p1&quot;]
for candidates in [[wrong], [wrong, correct]]:
    print([p[&quot;id&quot;] for p in candidates], any(p[&quot;model&quot;] == &quot;B&quot; for p in candidates))</code></pre><p>첫 경우는 검색 결과에서 필요한 자료가 빠졌고, 둘째 경우는 사용할 수 있는 근거가 있는데 생성 결과가 어긋났습니다. 둘째 경우까지 무조건 k를 키우면 원인과 다른 곳을 고칩니다.</p></details>

## 전이 과제 · 같은 후보를 받고 질문만 바꾸기

검색기가 같은 쪽을 돌려줘도 “언제 파일이 나뉘나요?”와 “파일이 나뉠 때 녹음이 멈추나요?”는 요구하는 답이 다릅니다. 생성 노드에 첫 후보의 본문만 넘기면 둘 다 같은 요약을 돌려줄 위험이 있습니다. 이번에는 후보를 고정하고 질문만 바꿉니다.

A형 1.1판 2쪽을 고정해 간격 질문과 상태 질문 두 개를 작성합니다. 두 실행 직전 생성 함수가 받은 (질문, 문서 ID) 쌍을 남겨 주세요. API 키가 있으면 같은 후보를 실제 답변 함수에도 보내 두 답의 핵심 문장을 비교하세요.

In [9]:
questions_to_try = ["A형은 몇 분마다 파일이 나뉘나요?", "A형은 파일이 바뀌면 녹음을 멈추나요?"]
# connect의 검색 함수가 항상 [target]을 주도록 하세요. 두 질문에서 답변 노드가 받은 질문과 쪽 ID를 출력하세요.

<details><summary>모범 예시와 풀이</summary><pre><code>def inspect_input(question, docs):
    return f&quot;{question} / {docs[0][&#x27;id&#x27;]}&quot;

fixed = connect(lambda question: [target], inspect_input)
for question in [&quot;A형은 몇 분마다 파일이 나뉘나요?&quot;, &quot;A형은 파일이 바뀌면 녹음을 멈추나요?&quot;]:
    print(fixed.invoke({&quot;question&quot;: question})[&quot;answer&quot;])
    if use_live:
        print(answer_from_documents(question, [target]))</code></pre><p>두 쌍의 ID는 같고 질문은 달라야 합니다. 직접 읽으면 각각 “45분마다”와 “녹음은 계속된다”가 핵심입니다. 실제 답이 두 값을 모두 언급해도 질문의 행동을 정확히 답하고 원문에 근거하면 됩니다.</p></details>

다음 장에서는 검색 후보가 도착한 뒤에도 생성으로 보낼지 한 번 판단합니다.